# 🎙️ AVSR Training — Audio-Visual Speech Recognition

**Мультимодальная модель**: Whisper encoder (аудио) + 3D-Conv/ResNet-18 (видео) + CrossAttention fusion + CTC.

**Ячейки:**
1. Проверка GPU
2. Google Drive
3. Установка зависимостей
4. Загрузка кода
5. Датасет
6. Манифесты
7. Конфиг
8. Модель
9. DataLoader'ы
10. Sanity check
11. 🚀 Обучение
12. TensorBoard
13. Оценка
14. Статус

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
else:
    print('⚠️  GPU не найден — Runtime → Change runtime type → T4 GPU')

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE_DIR    = '/content/drive/MyDrive/avsr_cursach'
CHECKPOINT_DIR = f'{DRIVE_DIR}/checkpoints'
DATA_DIR     = f'{DRIVE_DIR}/data'

for d in [CHECKPOINT_DIR, DATA_DIR, f'{DATA_DIR}/manifests',
          f'{DATA_DIR}/processed', f'{DATA_DIR}/audio']:
    os.makedirs(d, exist_ok=True)

print('✅ Drive подключён')
print('  Checkpoints:', CHECKPOINT_DIR)

In [ ]:
# Colab уже имеет torch/torchaudio/torchvision
!pip install -q \
    transformers==4.41.2 \
    omegaconf==2.3.0 \
    jiwer==3.0.4 \
    soundfile==0.12.1 \
    'mediapipe>=0.10.18' \
    opencv-python-headless \
    einops==0.8.0 \
    datasets
print('✅ Зависимости установлены')

In [ ]:
# ── Загрузка кода проекта ──────────────────────────────────
import os, sys

# Вариант A: git clone (вставь свой репозиторий)
# REPO_URL = 'https://github.com/YOUR/cursera_claude.git'
# !git clone $REPO_URL /content/avsr
# PROJECT_DIR = '/content/avsr'

# Вариант B: код уже в Google Drive (рекомендуется)
PROJECT_DIR = '/content/drive/MyDrive/avsr_cursach/code'
if not os.path.exists(os.path.join(PROJECT_DIR, 'src')):
    print('⚠️  Папка src/ не найдена в', PROJECT_DIR)
    print('Скопируй папку проекта в Google Drive → avsr_cursach/code/')
    print('Или раскомментируй git clone выше.')
else:
    if PROJECT_DIR not in sys.path:
        sys.path.insert(0, PROJECT_DIR)
    print('✅ Код загружен:', PROJECT_DIR)

In [ ]:
# ── Датасет: LJSpeech (аудио) для быстрого теста ─────────
# LJSpeech — 24h аудиокниг, нет видео, но подходит для проверки аудио-ветки
# Для полного AVSR нужен LRS3 или MUAVIC (видео+аудио)

from datasets import load_dataset
import soundfile as sf, json, os

MANIFEST_DIR = f'{DATA_DIR}/manifests'
AUDIO_DIR    = f'{DATA_DIR}/audio'
os.makedirs(AUDIO_DIR, exist_ok=True)

# Проверяем, не создан ли манифест раньше
if os.path.exists(f'{MANIFEST_DIR}/train.jsonl'):
    print('Манифесты уже готовы, пропускаем загрузку')
else:
    print('Загружаем LJSpeech (первые 500 примеров)...')
    ds = load_dataset('lj_speech', split='train[:500]', trust_remote_code=True)

    def make_manifest(samples, path):
        records = []
        for i, s in enumerate(samples):
            p = f'{AUDIO_DIR}/{i:05d}.wav'
            sf.write(p, s['audio']['array'], s['audio']['sampling_rate'])
            text = s.get('normalized_text', s.get('text','')).lower().strip()
            dur  = len(s['audio']['array']) / s['audio']['sampling_rate']
            if 0.5 <= dur <= 15.0 and text:
                records.append({'id':f'{i:05d}','audio':p,'video':'',
                                'lip_npy':'','text':text,'duration':round(dur,3)})
        with open(path,'w') as f:
            for r in records: f.write(json.dumps(r)+'\n')
        print(f'  {path}: {len(records)} примеров')

    make_manifest(ds.select(range(400)), f'{MANIFEST_DIR}/train.jsonl')
    make_manifest(ds.select(range(400,500)), f'{MANIFEST_DIR}/val.jsonl')
    print('✅ Манифесты готовы')

In [ ]:
from omegaconf import OmegaConf

cfg = OmegaConf.create({
    'experiment': {'name':'avsr_colab','seed':42,'output_dir': CHECKPOINT_DIR},
    'data': {
        'train_manifest': f'{DATA_DIR}/manifests/train.jsonl',
        'val_manifest':   f'{DATA_DIR}/manifests/val.jsonl',
        'max_duration': 15.0, 'min_duration': 0.5,
        'num_workers': 2, 'batch_size': 4, 'grad_accum': 4,
    },
    'audio': {'sample_rate':16000,'n_mels':80,'hop_length':160,'win_length':400},
    'video': {'fps':25,'lip_size':96},
    'model': {
        'mode': 'audio_only',  # ← меняй на 'av' когда будет видеодатасет
        'd_model': 512, 'modality_dropout': 0.1,
        'audio_encoder': {'name':'openai/whisper-small','freeze':True,'d_audio':768},
        'video_encoder': {'d_video':512,'n_layers':4,'n_heads':8},
        'fusion': {'type':'cross_attention','n_layers':2,'n_heads':8,'dropout':0.1},
    },
    'training': {
        'optimizer':'adamw','lr':1e-4,'weight_decay':0.01,
        'warmup_steps':200,'max_epochs':20,'patience':5,
        'grad_clip':5.0,'mixed_precision':True,'grad_accum':4,
    },
    'augmentation': {'audio':{'spec_augment':True,'freq_mask':15,'time_mask':10}},
    'logging': {'use_tensorboard':True,'use_wandb':False,
                'log_every_n_steps':20,'val_every_n_epochs':1},
})
print(OmegaConf.to_yaml(cfg))

In [ ]:
import torch, random
from src.data.dataset import CharTokenizer, AVSRDataset
from src.data.collate import avsr_collate_fn
from src.models.avsr_model import build_model
from torch.utils.data import DataLoader

torch.manual_seed(42); random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

tokenizer = CharTokenizer()
model = build_model(cfg, vocab_size=tokenizer.vocab_size).to(device)

n_all = sum(p.numel() for p in model.parameters())
n_tr  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Параметров всего: {n_all:,}  |  обучаемых: {n_tr:,}  |  заморожено: {n_all-n_tr:,}')

In [ ]:
train_ds = AVSRDataset(cfg.data.train_manifest, tokenizer,
    load_video=False, require_lip_cache=False)
val_ds   = AVSRDataset(cfg.data.val_manifest, tokenizer,
    load_video=False, require_lip_cache=False)

train_loader = DataLoader(train_ds, batch_size=cfg.data.batch_size,
    shuffle=True, num_workers=2, collate_fn=avsr_collate_fn,
    pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=cfg.data.batch_size,
    shuffle=False, num_workers=2, collate_fn=avsr_collate_fn, pin_memory=True)

print(f'Train: {len(train_ds)} примеров | Val: {len(val_ds)} примеров')

# Быстрая проверка батча
batch = next(iter(train_loader))
print('audio_mel:', batch['audio_mel'].shape, '| video:', batch['video'].shape)

In [ ]:
# Forward pass проверка
model.eval()
with torch.no_grad():
    b = {k: v.to(device) if isinstance(v, torch.Tensor) else v
         for k, v in batch.items()}
    logits, lens = model(b['audio_mel'], b['audio_lens'],
                         b['video'], b['video_lens'])
from src.training.metrics import decode_batch
preds = decode_batch(logits.cpu().float(), lens.cpu(), tokenizer)
print('Logits:', logits.shape, '| out_lens:', lens)
print('Предсказание (случайное):', preds[0][:60])
print('Reference:               ', batch['texts'][0][:60])
print('✅ Forward pass OK')

In [ ]:
# ============================================================
# 🚀 ЗАПУСК ОБУЧЕНИЯ
# ============================================================
import logging
logging.basicConfig(level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s', datefmt='%H:%M:%S')

from src.training.trainer import Trainer

trainer = Trainer(
    model=model, train_loader=train_loader, val_loader=val_loader,
    tokenizer=tokenizer, cfg=cfg, device=device, output_dir=CHECKPOINT_DIR
)

# Продолжить с последнего чекпоинта (если есть)
import os
last = os.path.join(CHECKPOINT_DIR, 'last.pt')
if os.path.exists(last):
    trainer.load_checkpoint(last)
    print(f'Продолжаю с epoch={trainer.start_epoch}')

trainer.fit()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {CHECKPOINT_DIR}/tb_logs

In [ ]:
# Статус — запускай в любой момент для проверки
import torch, os
from pathlib import Path
print('=== СТАТУС ОБУЧЕНИЯ ===')
for name in ['best.pt', 'last.pt']:
    p = Path(CHECKPOINT_DIR) / name
    if p.exists():
        ckpt = torch.load(str(p), map_location='cpu')
        print(f'[{name}] epoch={ckpt.get("epoch","?")}  '
              f'step={ckpt.get("global_step","?")}  '
              f'best_wer={ckpt.get("best_wer",float("nan")):.4f}')
    else:
        print(f'[{name}] — не найден')